In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%python
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyectofinaletl")

In [0]:
%python
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = "abfss://bronze@adlsproyectofinaletl.dfs.core.windows.net/circuits.csv"

In [0]:
%python

ventas_schema = StructType(fields=[StructField("fecha", StringType(), False),
                                     StructField("TipoCompr", StringType(), True),
                                     StructField("NumComp", StringType(), True),
                                     StructField("TipoDoc", StringType(), True),
                                     StructField("NumDoc", StringType(), True),
                                     StructField("cliente", StringType(), True),
                                     StructField("precio", DoubleType(), True),
                                     StructField("dscto", DoubleType(), True),
                                     StructField("total", DoubleType(), True)
])

In [0]:
%python
df_ventas_final = spark.read\
.option('header', True)\
.schema(circuits_schema)\
.csv('/path/to/your/file.csv')

In [0]:
%python
ventas_selected_df = (
    spark.read
    .option("header", True)
    .schema(ventas_schema)
    .csv(ruta)
    .select(
        col("TipoCompr"),
        col("NumComp"),
        col("TipoDoc"),
        col("NumDoc"),
        col("cliente"),
        col("precio"),
        col("dscto"),
        col("total")
    )
)

In [0]:
%python
ventas_renamed_df = ventas_selected_df.withColumnRenamed("TipoCompr", "tipo_comprobante") \
                                            .withColumnRenamed("NumComp", "nro_comprobante") \
                                            .withColumnRenamed("tipo_doc", "tipo_doc") \
                                            .withColumnRenamed("nro_doc", "num_doc") 

In [0]:
%python
ventas_final_df = ventas_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
%python
ventas_final_df = (
    spark.read
    .option("header", True)
    .schema(ventas_schema)
    .csv(ruta)
    .select(
        col("TipoCompr"),
        col("NumComp"),
        col("TipoDoc"),
        col("NumDoc"),
        col("cliente"),
        col("precio"),
        col("dscto"),
        col("total")
    )
    .withColumnRenamed("TipoCompr", "tipo_comprobante")
    .withColumnRenamed("NumComp", "nro_comprobante")
    .withColumnRenamed("TipoDoc", "tipo_doc")
    .withColumnRenamed("NumDoc", "num_doc")
    .withColumn("ingestion_date", current_timestamp())
)

ventas_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.ventas")